# Emerging Technologies Assessment

**Author:** Michael Ferry  
**Date:** January 2026  
**Module:** Emerging Technologies  

---

## Introduction

This notebook explores the difference between classical and quantum algorithms through the lens of the [Deutsch-Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa). The problems progress from generating Boolean functions and testing them classically, to building quantum oracles, implementing [Deutsch's algorithm](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm), and finally scaling up to the full Deutsch-Jozsa algorithm for four-bit inputs.

The Deutsch-Jozsa algorithm determines whether a Boolean function is constant or balanced using only **one query** to the function, whereas a classical computer would need up to $2^{n-1} + 1$ queries in the worst case. This exponential speedup is one of the earliest demonstrations of [quantum computational advantage](https://www.ibm.com/quantum/blog/group-theory).

All quantum circuits are implemented using [Qiskit](https://www.ibm.com/quantum/qiskit), IBM's open-source quantum computing framework, and simulated locally with [Qiskit Aer](https://github.com/Qiskit/qiskit-aer).

---

## Problem 1: Generating Random Boolean Functions

The [Deutsch-Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa) is designed to work with functions that accept a fixed number of [Boolean inputs](https://realpython.com/python-boolean/) and return a single Boolean output. Each function is guaranteed to be either **constant** (always returns `False` or always returns `True`) or **balanced** (returns `True` for exactly half of the possible input combinations).

The task is to write a Python function `random_constant_balanced` that returns a randomly chosen function from the set of constant or balanced functions taking four Boolean arguments as inputs.

### 1.1 Approach

With 4 Boolean inputs, there are $2^4 = 16$ possible input combinations. A constant function maps all 16 inputs to the same output, while a balanced function maps exactly 8 inputs to `True` and 8 to `False`. The strategy is to randomly decide the function type, then construct a lookup table accordingly. The function returned is a closure that captures this table.

In [1]:
import numpy as np
import random
from itertools import product

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

def random_constant_balanced():
    """
    Returns a randomly chosen constant or balanced Boolean function
    taking four Boolean arguments as inputs.
    
    The function randomly selects whether to generate a constant or
    balanced function, then constructs a lookup table mapping all
    16 possible 4-bit inputs to Boolean outputs.
    
    Returns
    -------
    callable
        A function f(a, b, c, d) -> bool where each argument is Boolean.
    """
    # All possible input combinations: 2^4 = 16 total
    all_inputs = list(product([False, True], repeat=4))
    
    # Randomly decide the function type with equal probability
    function_type = random.choice(['constant', 'balanced'])
    
    if function_type == 'constant':
        # Constant: every input maps to the same output value
        output_value = random.choice([False, True])
        lookup_table = {inputs: output_value for inputs in all_inputs}
    else:
        # Balanced: exactly 8 True and 8 False, randomly assigned
        outputs = [True] * 8 + [False] * 8
        random.shuffle(outputs)
        lookup_table = dict(zip(all_inputs, outputs))
    
    # Return a closure that looks up the precomputed answer
    def boolean_function(a, b, c, d):
        return lookup_table[(a, b, c, d)]
    
    return boolean_function

### 1.2 Verification Helper

To verify that our generated functions are valid, we need a helper that tests all 16 inputs and classifies the function. A valid constant function will have either 0 or 16 `True` outputs, while a valid balanced function will have exactly 8.

In [2]:
def verify_function_type(f):
    """
    Determines whether a Boolean function is constant or balanced
    by exhaustively testing all 16 possible 4-bit inputs.
    
    Parameters
    ----------
    f : callable
        A function taking 4 boolean arguments.
        
    Returns
    -------
    str
        'constant' if all outputs are identical,
        'balanced' if exactly 8 outputs are True,
        or an error description if neither condition holds.
    """
    all_inputs = list(product([False, True], repeat=4))
    results = [f(a, b, c, d) for a, b, c, d in all_inputs]
    true_count = sum(results)
    
    if true_count == 0 or true_count == 16:
        return 'constant'
    elif true_count == 8:
        return 'balanced'
    else:
        return f'invalid ({true_count}/16 are True)'

### 1.3 Testing the Generator

Let's generate several functions and verify they are all valid constant or balanced functions. We also display a complete truth table for one example to illustrate the structure.

In [3]:
# Generate and classify 10 random functions
print("Generating 10 random Boolean functions:")
print("-" * 50)
for i in range(10):
    f = random_constant_balanced()
    ftype = verify_function_type(f)
    
    # Count True outputs to show the split
    all_inputs = list(product([False, True], repeat=4))
    true_count = sum(f(a, b, c, d) for a, b, c, d in all_inputs)
    print(f"  Function {i+1:2d}: {ftype:>8s}  (True count: {true_count}/16)")

Generating 10 random Boolean functions:
--------------------------------------------------
  Function  1: constant  (True count: 0/16)
  Function  2: balanced  (True count: 8/16)
  Function  3: constant  (True count: 16/16)
  Function  4: constant  (True count: 16/16)
  Function  5: balanced  (True count: 8/16)
  Function  6: balanced  (True count: 8/16)
  Function  7: constant  (True count: 0/16)
  Function  8: constant  (True count: 16/16)
  Function  9: constant  (True count: 0/16)
  Function 10: constant  (True count: 16/16)


In [4]:
# Display a complete truth table for one function
print("=" * 60)
print("EXAMPLE: Complete truth table for a generated function")
print("=" * 60)

f = random_constant_balanced()
all_inputs = list(product([False, True], repeat=4))

print(f"\nFunction type: {verify_function_type(f)}")
print("\nComplete truth table:")
print("-" * 60)
print(f"{'Input (a, b, c, d)':<30} | Output")
print("-" * 60)

for inputs in all_inputs:
    a, b, c, d = inputs
    output = f(a, b, c, d)
    print(f"({a!s:5}, {b!s:5}, {c!s:5}, {d!s:5})       | {output}")

print("-" * 60)
true_count = sum(f(a, b, c, d) for a, b, c, d in all_inputs)
print(f"Total True outputs: {true_count}/16")

EXAMPLE: Complete truth table for a generated function

Function type: balanced

Complete truth table:
------------------------------------------------------------
Input (a, b, c, d)             | Output
------------------------------------------------------------
(False, False, False, False)       | False
(False, False, False, True )       | True
(False, False, True , False)       | False
(False, False, True , True )       | True
(False, True , False, False)       | True
(False, True , False, True )       | True
(False, True , True , False)       | False
(False, True , True , True )       | True
(True , False, False, False)       | True
(True , False, False, True )       | False
(True , False, True , False)       | False
(True , False, True , True )       | True
(True , True , False, False)       | True
(True , True , False, True )       | False
(True , True , True , False)       | False
(True , True , True , True )       | False
-------------------------------------------------------

### 1.4 Discussion

With 4 Boolean inputs, there are $2^{16} = 65{,}536$ total possible Boolean functions. Of these, only 2 are constant (all-`True` or all-`False`) and $\binom{16}{8} = 12{,}870$ are balanced (exactly 8 `True` outputs). The generator uses [`itertools.product`](https://docs.python.org/3/library/itertools.html#itertools.product) to enumerate all input combinations and [`random.shuffle`](https://docs.python.org/3/library/random.html#random.shuffle) to create balanced distributions.

The lookup-table approach mirrors how quantum oracles work: they are precomputed mappings between computational basis states (Nielsen and Chuang, 2010, Ch. 1.4.5).

### 1.5 References

- **Deutsch, D. and Jozsa, R. (1992)** 'Rapid Solution of Problems by Quantum Computation', *Proceedings of the Royal Society of London. Series A*, 439(1907), pp. 553–558. Available at: [https://doi.org/10.1098/rspa.1992.0167](https://doi.org/10.1098/rspa.1992.0167). Section 2 defines constant and balanced Boolean functions.
- **Nielsen, M.A. and Chuang, I.L. (2010)** *Quantum Computation and Quantum Information: 10th Anniversary Edition*. Cambridge: Cambridge University Press. Chapter 1.4.5 explains the lookup-table approach and its connection to quantum oracles.
- **Python Software Foundation (2024)** *itertools — Functions creating iterators for efficient looping*. Available at: [https://docs.python.org/3/library/itertools.html](https://docs.python.org/3/library/itertools.html).

---

## Problem 2: Classical Testing for Function Type

To appreciate the advantage of quantum algorithms, we first need to understand the classical approach. Given a function (like those from Problem 1), a classical algorithm must call the function with different inputs and analyze the results to determine whether it is constant or balanced.

The key question is: **how many function calls are needed to be 100% certain?**

### 2.1 Theoretical Analysis

For $n = 4$ input bits, there are $2^4 = 16$ possible inputs. A balanced function has exactly 8 `True` and 8 `False` outputs. Therefore, if we observe 9 identical outputs in a row, the function **must** be constant — the remaining 7 outputs cannot produce the required 8-8 split for a balanced function.

This gives us a worst-case complexity of $2^{n-1} + 1 = 9$ queries. In the best case, we might find two different outputs on the very first two queries, immediately proving the function is balanced.

In [5]:
def determine_constant_balanced(f):
    """
    Determines whether a function is constant or balanced using
    a classical approach: testing inputs sequentially.
    
    The algorithm stops as soon as it can make a definitive determination:
    - If two different outputs are found, the function must be balanced.
    - If more than half the inputs give the same output, it must be constant.
    
    Parameters
    ----------
    f : callable
        A Boolean function taking 4 arguments.
        
    Returns
    -------
    tuple of (str, int)
        The function type ('constant' or 'balanced') and the number
        of function calls made to reach the conclusion.
    """
    all_inputs = list(product([False, True], repeat=4))
    
    # Evaluate the first input as our reference
    first_result = f(*all_inputs[0])
    calls = 1
    
    for inputs in all_inputs[1:]:
        result = f(*inputs)
        calls += 1
        
        # Found a different result => must be balanced
        if result != first_result:
            return 'balanced', calls
        
        # Checked more than half with identical results => must be constant
        if calls > len(all_inputs) // 2:
            return 'constant', calls
    
    return 'constant', calls

### 2.2 Testing the Classical Algorithm

Let's verify that the classical algorithm correctly identifies function types, and observe how the number of queries varies between constant and balanced functions.

In [6]:
# Verify correctness and measure query counts
print("Testing classical algorithm on 10 random functions:")
print("-" * 65)
print(f"{'Test':<6} {'Actual':<12} {'Determined':<12} {'Queries':<8}")
print("-" * 65)

for i in range(10):
    f = random_constant_balanced()
    actual_type = verify_function_type(f)
    determined_type, num_calls = determine_constant_balanced(f)
    match = "✓" if actual_type == determined_type else "✗"
    print(f"  {i+1:<4} {actual_type:<12} {determined_type:<12} {num_calls:<8} {match}")

Testing classical algorithm on 10 random functions:
-----------------------------------------------------------------
Test   Actual       Determined   Queries 
-----------------------------------------------------------------
  1    constant     constant     9        ✓
  2    balanced     balanced     2        ✓
  3    constant     constant     9        ✓
  4    constant     constant     9        ✓
  5    balanced     balanced     5        ✓
  6    constant     constant     9        ✓
  7    constant     constant     9        ✓
  8    constant     constant     9        ✓
  9    constant     constant     9        ✓
  10   balanced     balanced     3        ✓


### 2.3 Worst-Case Demonstration

The worst case occurs when testing a constant function: we must check $2^{n-1} + 1 = 9$ inputs before we can be certain. Let's trace through this step by step.

In [7]:
# Generate a constant function for worst-case analysis
f = random_constant_balanced()
while verify_function_type(f) != 'constant':
    f = random_constant_balanced()

print("=" * 70)
print("WORST-CASE DEMONSTRATION: Testing a constant function")
print("=" * 70)

all_inputs = list(product([False, True], repeat=4))
first_result = f(*all_inputs[0])
print(f"\nQuery 1: f{all_inputs[0]} = {first_result}")
print(f"         Status: Need more data\n")

for i in range(1, 9):
    result = f(*all_inputs[i])
    print(f"Query {i+1}: f{all_inputs[i]} = {result}")
    
    if i < 8:
        print(f"         Status: {i+1} matching results, still uncertain\n")
    else:
        print(f"         Status: 9 matching results → DEFINITELY CONSTANT")
        print(f"\n  Reasoning: A balanced function must have 8 True and 8 False.")
        print(f"  With 9 identical outputs, the remaining 7 cannot balance it.")

WORST-CASE DEMONSTRATION: Testing a constant function

Query 1: f(False, False, False, False) = True
         Status: Need more data

Query 2: f(False, False, False, True) = True
         Status: 2 matching results, still uncertain

Query 3: f(False, False, True, False) = True
         Status: 3 matching results, still uncertain

Query 4: f(False, False, True, True) = True
         Status: 4 matching results, still uncertain

Query 5: f(False, True, False, False) = True
         Status: 5 matching results, still uncertain

Query 6: f(False, True, False, True) = True
         Status: 6 matching results, still uncertain

Query 7: f(False, True, True, False) = True
         Status: 7 matching results, still uncertain

Query 8: f(False, True, True, True) = True
         Status: 8 matching results, still uncertain

Query 9: f(True, False, False, False) = True
         Status: 9 matching results → DEFINITELY CONSTANT

  Reasoning: A balanced function must have 8 True and 8 False.
  With 9 id

### 2.4 Statistical Comparison

Let's compare how the classical algorithm performs across many random functions to see the distribution of query counts.

In [8]:
# Statistical comparison across 20 random functions
print("=" * 70)
print("COMPARATIVE ANALYSIS: Query efficiency across 20 random functions")
print("=" * 70)

constant_calls = []
balanced_calls = []

for i in range(20):
    f = random_constant_balanced()
    actual_type = verify_function_type(f)
    determined_type, num_calls = determine_constant_balanced(f)
    
    if actual_type == 'constant':
        constant_calls.append(num_calls)
    else:
        balanced_calls.append(num_calls)

print(f"\nConstant functions tested: {len(constant_calls)}")
if constant_calls:
    print(f"  Query counts: {constant_calls}")
    print(f"  All required exactly 9 queries (worst case)")

print(f"\nBalanced functions tested: {len(balanced_calls)}")
if balanced_calls:
    print(f"  Query counts: {balanced_calls}")
    print(f"  Min: {min(balanced_calls)}, Max: {max(balanced_calls)}, "
          f"Avg: {sum(balanced_calls)/len(balanced_calls):.1f}")

print("\n" + "=" * 70)
print("KEY INSIGHT:")
print("  Classical worst case: 9 queries  (O(2^(n-1) + 1))")
print("  Quantum (Deutsch-Jozsa): 1 query (O(1))")
print("=" * 70)

COMPARATIVE ANALYSIS: Query efficiency across 20 random functions

Constant functions tested: 12
  Query counts: [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]
  All required exactly 9 queries (worst case)

Balanced functions tested: 8
  Query counts: [2, 7, 2, 2, 2, 6, 2, 2]
  Min: 2, Max: 7, Avg: 3.1

KEY INSIGHT:
  Classical worst case: 9 queries  (O(2^(n-1) + 1))
  Quantum (Deutsch-Jozsa): 1 query (O(1))


### 2.5 Efficiency Summary

The classical algorithm's performance depends on the function type:

| Scenario | Queries | Explanation |
|----------|---------|-------------|
| **Best case** | 2 | Two different outputs found immediately → balanced |
| **Worst case** | 9 | Must check $2^{n-1} + 1 = 9$ identical outputs to confirm constant |
| **Average** | ~5–6 | Balanced functions detected quickly; constant functions always need 9 |

The classical query complexity is $O(2^{n-1} + 1)$ where $n$ is the number of input bits. The quantum Deutsch–Jozsa algorithm solves this with exactly **1 query**, regardless of function type or the number of input bits.

### 2.6 References

- **Cleve, R., Ekert, A., Macchiavello, C. and Mosca, M. (1998)** 'Quantum algorithms revisited', *Proceedings of the Royal Society of London. Series A*, 454(1969), pp. 339–354. Available at: [https://doi.org/10.1098/rspa.1998.0164](https://doi.org/10.1098/rspa.1998.0164). Proves the $2^{n-1}+1$ classical lower bound.
- **Deutsch, D. and Jozsa, R. (1992)** 'Rapid Solution of Problems by Quantum Computation', *Proceedings of the Royal Society of London. Series A*, 439(1907), pp. 553–558. Available at: [https://doi.org/10.1098/rspa.1992.0167](https://doi.org/10.1098/rspa.1992.0167).

---
## Problem 3: Quantum Oracles

**Date:** March 2026

Deutsch's algorithm is the simplest quantum algorithm that demonstrates quantum advantage through superposition. It works with Boolean functions that take a single input bit.

With one Boolean input, there are exactly **4 possible Boolean functions**:

1. **f(x) = 0** (constant, always returns False)
2. **f(x) = 1** (constant, always returns True)  
3. **f(x) = x** (balanced, identity function)
4. **f(x) = NOT x** (balanced, negation function)

A **quantum oracle** is a unitary operator that encodes a classical function into a quantum circuit. The oracle applies the transformation:

|x⟩|y⟩ → |x⟩|y ⊕ f(x)⟩

Where ⊕ represents XOR (addition modulo 2).

In [9]:
# Quantum computing imports
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

In [10]:
# Oracle 1: f(x) = 0 (constant and always returns false)
def oracle_constant_0():
    """
    Creates a quantum oracle for f(x) = 0.
    Since the function always returns 0, we don't apply any gates.
    The oracle does nothing - implements |x⟩|y⟩ → |x⟩|y⟩
    """
    qc = QuantumCircuit(2, name='f(x)=0')
    # No gates needed, function always returns 0
    return qc

# Tests and visualizes
oracle = oracle_constant_0()
print("Oracle for f(x) = 0 (constant):")
print(oracle.draw())

Oracle for f(x) = 0 (constant):
     
q_0: 
     
q_1: 
     


In [11]:
# Oracle 2: f(x) = 1 (constant and always returns true)
def oracle_constant_1():
    """
    Creates a quantum oracle for f(x) = 1.
    Since function always returns 1, we flip the output qubit.
    Implements |x⟩|y⟩ → |x⟩|y ⊕ 1⟩
    """
    qc = QuantumCircuit(2, name='f(x)=1')
    # Apply X gate to output qubit (qubit 1)
    qc.x(1)
    return qc

# Test and visualize
oracle = oracle_constant_1()
print("\nOracle for f(x) = 1 (constant):")
print(oracle.draw())


Oracle for f(x) = 1 (constant):
          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘


In [12]:
# Oracle 3: f(x) = x (balanced, identity function)
def oracle_identity():
    """
    Creates a quantum oracle for f(x) = x.
    Output equals input, so we use CNOT with input as control.
    Implements |x⟩|y⟩ → |x⟩|y ⊕ x⟩
    """
    qc = QuantumCircuit(2, name='f(x)=x')
    # CNOT: control=qubit 0 (input), target=qubit 1 (output)
    qc.cx(0, 1)
    return qc

# Test and visualise
oracle = oracle_identity()
print("\nOracle for f(x) = x (balanced, identity):")
print(oracle.draw())


Oracle for f(x) = x (balanced, identity):
          
q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘


In [13]:
# Oracle 4: f(x) = NOT x (balanced, negation function)
def oracle_negation():
    """
    Creates a quantum oracle for f(x) = NOT x.
    Output is opposite of input: flip input then CNOT, or CNOT then flip output.
    Implements |x⟩|y⟩ → |x⟩|y ⊕ NOT x⟩
    """
    qc = QuantumCircuit(2, name='f(x)=NOT x')
    # Apply X to input, then CNOT, then X back to input
    # This effectively does: output ⊕= NOT input
    qc.x(0)
    qc.cx(0, 1)
    qc.x(0)
    return qc

# Tests and visualise
oracle = oracle_negation()
print("\nOracle for f(x) = NOT x (balanced, negation):")
print(oracle.draw())


Oracle for f(x) = NOT x (balanced, negation):
     ┌───┐     ┌───┐
q_0: ┤ X ├──■──┤ X ├
     └───┘┌─┴─┐└───┘
q_1: ─────┤ X ├─────
          └───┘     


### Oracle Implementation Notes

Each oracle implements the transformation |x⟩|y⟩ → |x⟩|y ⊕ f(x)⟩ where:
- Qubit 0 is the input x
- Qubit 1 is the output y
- ⊕ is XOR (addition modulo 2)

**Constant oracles:**
- **f(x) = 0**: No gates needed - output stays unchanged
- **f(x) = 1**: X gate on output qubit - always flips it

**Balanced oracles:**
- **f(x) = x**: CNOT gate copies input to output via XOR
- **f(x) = NOT x**: X-CNOT-X sequence - flips input first, then copies, then flips back

These four oracles represent all possible single-bit Boolean functions and will be used in Deutsch's algorithm in Problem 4.

### References

**Deutsch, D. (1985)** 'Quantum theory, the Church-Turing principle and the universal quantum computer', *Proceedings of the Royal Society of London A: Mathematical, Physical and Engineering Sciences*, 400(1818), pp. 97-117.

This is Deutsch's original paper introducing the first quantum algorithm. Section 5 discusses quantum gates and how classical functions can be encoded as reversible quantum operations, which is the foundation for quantum oracles.

**Nielsen, M.A. and Chuang, I.L. (2010)** *Quantum Computation and Quantum Information: 10th Anniversary Edition*. Cambridge: Cambridge University Press.

Chapter 1.4.4 explains quantum oracles in detail, including how the phase kickback technique works. Section 1.4.5 covers Deutsch's algorithm and demonstrates how these four oracles are used to determine if a function is constant or balanced with a single query.

**Qiskit Documentation (2024)** *Building Quantum Circuits*. Available at: https://docs.quantum.ibm.com/api/qiskit/circuit

Official documentation for the QuantumCircuit class and gate operations used to implement these oracles. The CNOT (cx) and X gates are fundamental building blocks for quantum algorithms.

---
## Problem 4: Deutsch's Algorithm with Qiskit

**Date:** March 2026

Deutsch's algorithm is the simplest quantum algorithm that demonstrates quantum advantage. It determines whether a single input Boolean function is constant or balanced using just **one query** to the oracle.

The algorithm works by:
1. Preparing qubits in superposition using Hadamard gates
2. Applying the oracle (from Problem 3)
3. Applying another Hadamard gate to create interference
4. Measuring the result

If the measurement is **0**, the function is constant.  
If the measurement is **1**, the function is balanced.

This is exponentially better than the classical approach, which needs 2 queries in the worst case for a single-input function (and 2^(n-1) + 1 queries for n inputs).

In [14]:
def build_deutsch_circuit(oracle):
    """
    Implements Deutsch's Algorithm for a given 2-qubit oracle.
    
    Parameters:
    -----------
    oracle : QuantumCircuit
        A 2-qubit quantum circuit representing the function f(x).
        
    Returns:
    --------
    QuantumCircuit
        The full Deutsch algorithm circuit ready for execution.
    """
    # Create a quantum circuit with 2 qubits and 1 classical bit for measurement
    qc = QuantumCircuit(2, 1, name="Deutsch")
    
    # Step 1: Initialize the output qubit (qubit 1) to state |1>
    # (Input qubit 0 defaults to |0>, which is what we want)
    qc.x(1)
    
    # Step 2: Apply Hadamard gates to both qubits to create superposition
    qc.h(0)
    qc.h(1)
    
    # Add a barrier to visually separate the algorithm steps from the oracle
    qc.barrier()
    
    # Step 3: Append the oracle to our circuit
    qc.compose(oracle, inplace=True)
    
    # Add another barrier after the oracle
    qc.barrier()
    
    # Step 4: Apply a final Hadamard gate to the input qubit (qubit 0)
    # This creates the interference pattern we need
    qc.h(0)
    
    # Step 5: Measure the input qubit and store result in the classical bit
    qc.measure(0, 0)
    
    return qc

# Let's quickly visualize what the empty framework looks like 
# using our constant_0 oracle from Problem 3
print("Deutsch's Algorithm Circuit Architecture:")
display(build_deutsch_circuit(oracle_constant_0()).draw('text'))

Deutsch's Algorithm Circuit Architecture:


┌───┐      ░  ░ ┌───┐┌─┐
q_0: ┤ H ├──────░──░─┤ H ├┤M├
     ├───┤┌───┐ ░  ░ └───┘└╥┘
q_1: ┤ X ├┤ H ├─░──░───────╫─
     └───┘└───┘ ░  ░       ║ 
c: 1/══════════════════════╩═
                           0

### 4.1 Executing the Algorithm: Constant Functions

We will use the `qasm_simulator` from Qiskit Aer to execute our quantum circuits. 

According to Deutsch's algorithm, if the function is constant, the interference pattern will cause the final measurement to **always be 0** (100% probability). Let's test both of our constant oracles to verify this behavior.

In [15]:
# Import required Qiskit modules
from qiskit import QuantumCircuit
from qiskit_aer import Aer

# Initialize the local simulator
simulator = Aer.get_backend('qasm_simulator')

print("Testing Constant Functions")
print("=" * 40)

# 1. Test the oracle for f(x) = 0
qc_const_0 = build_deutsch_circuit(oracle_constant_0())
job_0 = simulator.run(qc_const_0, shots=1024)
result_0 = job_0.result().get_counts()

print("Oracle 1: f(x) = 0")
print(f"Measurement counts: {result_0}")
print("Conclusion: Function is Constant (Measured '0')\n")

# 2. Test the oracle for f(x) = 1
qc_const_1 = build_deutsch_circuit(oracle_constant_1())
job_1 = simulator.run(qc_const_1, shots=1024)
result_1 = job_1.result().get_counts()

print("Oracle 2: f(x) = 1")
print(f"Measurement counts: {result_1}")
print("Conclusion: Function is Constant (Measured '0')")

Testing Constant Functions
Oracle 1: f(x) = 0
Measurement counts: {'0': 1024}
Conclusion: Function is Constant (Measured '0')

Oracle 2: f(x) = 1
Measurement counts: {'0': 1024}
Conclusion: Function is Constant (Measured '0')


### 4.2 Executing the Algorithm: Balanced Functions

For a balanced function, the phase kickback will alter the state such that the final measurement will **always be 1** (100% probability).

In [16]:
print("Testing Balanced Functions")
print("=" * 40)

# 3. Test the identity oracle f(x) = x
qc_balanced_id = build_deutsch_circuit(oracle_identity())
job_id = simulator.run(qc_balanced_id, shots=1024)
result_id = job_id.result().get_counts()

print("Oracle 3: f(x) = x (Identity)")
print(f"Measurement counts: {result_id}")
print("Conclusion: Function is Balanced (Measured '1')\n")

# 4. Test the negation oracle f(x) = NOT x
qc_balanced_neg = build_deutsch_circuit(oracle_negation())
job_neg = simulator.run(qc_balanced_neg, shots=1024)
result_neg = job_neg.result().get_counts()

print("Oracle 4: f(x) = NOT x (Negation)")
print(f"Measurement counts: {result_neg}")
print("Conclusion: Function is Balanced (Measured '1')")

Testing Balanced Functions
Oracle 3: f(x) = x (Identity)
Measurement counts: {'1': 1024}
Conclusion: Function is Balanced (Measured '1')

Oracle 4: f(x) = NOT x (Negation)
Measurement counts: {'1': 1024}
Conclusion: Function is Balanced (Measured '1')


### 4.3 Understanding the Interference Pattern

How does this work with just one query? 

1. **Initialization:** The initial Hadamard gates put the input qubit into a superposition state of $|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$ and the output qubit into $|-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$.
2. **Phase Kickback:** When the oracle is applied, a phenomenon called *phase kickback* occurs. If the function evaluates to 1, the phase of the $|-\rangle$ state is flipped (multiplied by -1). 
3. **Constructive/Destructive Interference:** * For **constant functions**, the phase is either unchanged or flipped globally, leaving the input qubit in the $|+\rangle$ state. The final Hadamard gate turns this back into exactly $|0\rangle$.
    * For **balanced functions**, the phase is flipped for exactly half of the input superposition. This changes the input qubit's state to $|-\rangle$. The final Hadamard gate turns this into exactly $|1\rangle$.

This interference pattern is what allows the algorithm to determine the function's global property in a single evaluation, completely bypassing the need to check individual inputs like a classical computer would.

### 4.4 References and Implementation Notes

**IBM Quantum Learning (2024)** *Fundamentals of Quantum Algorithms: Deutsch's Algorithm*. Available at: https://quantum.cloud.ibm.com/learning/course/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm

This official IBM module provides the theoretical foundation and the Qiskit-specific implementation details for Deutsch's algorithm. It was instrumental in designing the `build_deutsch_circuit` function, specifically detailing why the output qubit must be initialized to the $|1\rangle$ state before applying the Hadamard gate to properly trigger the phase kickback mechanism.

**Nielsen, M.A. and Chuang, I.L. (2010)** *Quantum Computation and Quantum Information: 10th Anniversary Edition*. Cambridge: Cambridge University Press.

Chapter 1.4.5 provides a comprehensive mathematical breakdown of the algorithm. The textbook explains the "Hadamard sandwich" technique used in this problem's circuit: applying $H$ gates to create a superposition, applying the oracle $U_f$, and applying a final $H$ gate to produce interference. This reference mathematically proves why the state resolves to exactly $|0\rangle$ for constant functions and $|1\rangle$ for balanced functions.

**Cleve, R., Ekert, A., Macchiavello, C., and Mosca, M. (1998)** 'Quantum algorithms revisited', *Proceedings of the Royal Society of London. Series A: Mathematical, Physical and Engineering Sciences*, 454(1969), pp. 339-354. Available at: https://doi.org/10.1098/rspa.1998.0164

This paper formalized the modern quantum circuit representation of Deutsch's algorithm used in this implementation. While Deutsch's original 1985 paper only succeeded with a 50% probability, Cleve et al. introduced the phase kickback technique (using the $|-\rangle$ state on the target qubit) that allows the circuit to succeed with 100% certainty in a single query.

---

## Problem 5: Scaling to the Deutsch-Jozsa Algorithm

The [Deutsch-Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa) generalizes Deutsch's approach from a single-input function to functions with $n$ input bits. For the four-bit functions generated in Problem 1, this means we can determine whether a function is constant or balanced with a **single quantum query** compared to the classical worst case of 9 queries.

### 5.1 How It Works

The algorithm follows the same principle as Deutsch's algorithm but with $n$ input qubits instead of 1:

1. **Initialize** $n$ input qubits to $|0\rangle$ and one output qubit to $|1\rangle$
2. **Apply Hadamard gates** to all $n + 1$ qubits
3. **Apply the oracle** $U_f$ (one query)
4. **Apply Hadamard gates** to the $n$ input qubits
5. **Measure** the input qubits: if all are $|0\rangle$, the function is constant; otherwise, it is balanced


### 5.2 Encoding Classical Functions as Quantum Oracles

To use the four-bit Boolean functions from Problem 1 in a quantum circuit, we need to encode them as unitary transformations. The oracle implements:

$$|x_1 x_2 x_3 x_4\rangle|y\rangle \rightarrow |x_1 x_2 x_3 x_4\rangle|y \oplus f(x_1, x_2, x_3, x_4)\rangle$$

We do this by iterating over all 16 possible inputs. For each input where $f$ returns `True`, we add a multi-controlled X gate (MCX) that flips the output qubit only when the input qubits match that specific combination. The X gates before and after the MCX ensure that the control pattern matches the desired input (by temporarily flipping qubits that should be $|0\rangle$ in the target pattern).

In [17]:
def build_dj_oracle(f, n=4):
    """
    Builds a quantum oracle for an n-bit Boolean function.
    
    For each input combination where f returns True, a multi-controlled
    X gate is added to flip the output qubit. X gates are used to
    condition on |0> states by temporarily flipping those qubits.
    
    Parameters
    ----------
    f : callable
        A Boolean function taking n boolean arguments.
    n : int
        Number of input bits (default 4).
        
    Returns
    -------
    QuantumCircuit
        An (n+1)-qubit oracle circuit.
    """
    num_qubits = n + 1  # n input qubits + 1 output qubit
    oracle = QuantumCircuit(num_qubits, name='Oracle')
    
    # Iterate over all possible n-bit input combinations
    all_inputs = list(product([False, True], repeat=n))
    
    for input_combo in all_inputs:
        # Only add gates for inputs where f returns True
        if f(*input_combo):
            # Apply X gates to qubits that should be |0> in this input
            # This makes the multi-controlled gate trigger on the right pattern
            for i, bit in enumerate(input_combo):
                if not bit:
                    oracle.x(i)
            
            # Multi-controlled X: flip output qubit when all inputs match
            oracle.mcx(list(range(n)), n)
            
            # Undo the X gates to restore the input qubits
            for i, bit in enumerate(input_combo):
                if not bit:
                    oracle.x(i)
    
    return oracle

### 5.3 The Deutsch-Jozsa Circuit

Now we build the full Deutsch-Jozsa circuit that wraps the oracle in Hadamard gates and measures the result.

In [18]:
def build_deutsch_jozsa_circuit(f, n=4):
    """
    Implements the Deutsch-Jozsa algorithm for an n-bit function.
    
    Parameters
    ----------
    f : callable
        A Boolean function taking n boolean arguments.
    n : int
        Number of input bits (default 4).
        
    Returns
    -------
    QuantumCircuit
        The complete Deutsch-Jozsa circuit.
    """
    num_qubits = n + 1
    qc = QuantumCircuit(num_qubits, n)
    
    # Step 1: Initialize output qubit to |1>
    qc.x(n)
    
    # Step 2: Apply Hadamard to all qubits
    for i in range(num_qubits):
        qc.h(i)
    
    qc.barrier()
    
    # Step 3: Apply the oracle (single query)
    oracle = build_dj_oracle(f, n)
    qc.compose(oracle, inplace=True)
    
    qc.barrier()
    
    # Step 4: Apply Hadamard to input qubits only
    for i in range(n):
        qc.h(i)
    
    # Step 5: Measure input qubits
    for i in range(n):
        qc.measure(i, i)
    
    return qc

### 5.4 Demonstrating with Constant Functions

For a constant function, the Deutsch-Jozsa algorithm should always measure $|0000\rangle$ meaning all input qubits return to $|0\rangle$ after the final Hadamard gates.

In [19]:
simulator = Aer.get_backend('qasm_simulator')

print("=" * 70)
print("DEUTSCH-JOZSA: Testing Constant Functions")
print("=" * 70)

# Constant function: f(...) = False for all inputs
def const_false(a, b, c, d):
    return False

qc_cf = build_deutsch_jozsa_circuit(const_false)
result_cf = simulator.run(qc_cf, shots=1024).result().get_counts()
print(f"\n  f = constant False")
print(f"  Measurement: {result_cf}")
is_constant = '0000' in result_cf and len(result_cf) == 1
print(f"  All zeros? {is_constant}  →  Correctly identified as CONSTANT")

# Constant function: f(...) = True for all inputs
def const_true(a, b, c, d):
    return True

qc_ct = build_deutsch_jozsa_circuit(const_true)
result_ct = simulator.run(qc_ct, shots=1024).result().get_counts()
print(f"\n  f = constant True")
print(f"  Measurement: {result_ct}")
is_constant = '0000' in result_ct and len(result_ct) == 1
print(f"  All zeros? {is_constant}  →  Correctly identified as CONSTANT")


DEUTSCH-JOZSA: Testing Constant Functions

  f = constant False
  Measurement: {'0000': 1024}
  All zeros? True  →  Correctly identified as CONSTANT

  f = constant True
  Measurement: {'0000': 1024}
  All zeros? True  →  Correctly identified as CONSTANT


### 5.5 Demonstrating with Balanced Functions

For balanced functions, the measurement should produce **any result other than** $|0000\rangle$. Let's test with two specific balanced functions: one where the first input bit determines the output, and one using a parity-based rule.

In [20]:
print("=" * 70)
print("DEUTSCH-JOZSA: Testing Balanced Functions")
print("=" * 70)

# Balanced function 1: f(a,b,c,d) = a (output equals first input)
def balanced_first_bit(a, b, c, d):
    return a

qc_b1 = build_deutsch_jozsa_circuit(balanced_first_bit)
result_b1 = simulator.run(qc_b1, shots=1024).result().get_counts()
print(f"\n  f(a,b,c,d) = a  (first bit determines output)")
print(f"  Measurement: {result_b1}")
is_balanced = '0000' not in result_b1
print(f"  Non-zero result? {is_balanced}  →  Correctly identified as BALANCED")

# Balanced function 2: f(a,b,c,d) = a XOR b (parity of first two bits)
def balanced_parity(a, b, c, d):
    return a ^ b

qc_b2 = build_deutsch_jozsa_circuit(balanced_parity)
result_b2 = simulator.run(qc_b2, shots=1024).result().get_counts()
print(f"\n  f(a,b,c,d) = a XOR b  (parity of first two bits)")
print(f"  Measurement: {result_b2}")
is_balanced = '0000' not in result_b2
print(f"  Non-zero result? {is_balanced}  →  Correctly identified as BALANCED")

DEUTSCH-JOZSA: Testing Balanced Functions

  f(a,b,c,d) = a  (first bit determines output)
  Measurement: {'0001': 1024}
  Non-zero result? True  →  Correctly identified as BALANCED

  f(a,b,c,d) = a XOR b  (parity of first two bits)
  Measurement: {'0011': 1024}
  Non-zero result? True  →  Correctly identified as BALANCED


### 5.6 Testing with Random Functions from Problem 1

Finally, let's use the `random_constant_balanced` generator from Problem 1 to test the algorithm on randomly generated functions, and compare the quantum result with the classical verification.

In [21]:
print("=" * 70)
print("DEUTSCH-JOZSA vs CLASSICAL: Testing random functions from Problem 1")
print("=" * 70)
print(f"\n{'#':<4} {'Classical':>10} {'Quantum':>10} {'Match':>6}")
print("-" * 35)

# Reset seed so we get fresh random functions
random.seed(123)

all_correct = True
for i in range(10):
    f = random_constant_balanced()
    
    # Classical verification (exhaustive)
    classical_type = verify_function_type(f)
    
    # Quantum determination (single query via Deutsch-Jozsa)
    qc = build_deutsch_jozsa_circuit(f)
    counts = simulator.run(qc, shots=1024).result().get_counts()
    quantum_type = 'constant' if '0000' in counts and len(counts) == 1 else 'balanced'
    
    match = "✓" if classical_type == quantum_type else "✗"
    if classical_type != quantum_type:
        all_correct = False
    print(f"{i+1:<4} {classical_type:>10} {quantum_type:>10} {match:>6}")

print("-" * 35)
print(f"All results match: {all_correct}")
print(f"\nThe quantum algorithm used exactly 1 oracle query per function,")
print(f"while the classical algorithm needed up to 9 queries.")

DEUTSCH-JOZSA vs CLASSICAL: Testing random functions from Problem 1

#     Classical    Quantum  Match
-----------------------------------
1      constant   constant      ✓
2      constant   constant      ✓
3      balanced   balanced      ✓
4      constant   constant      ✓
5      constant   constant      ✓
6      constant   constant      ✓
7      constant   constant      ✓
8      balanced   balanced      ✓
9      balanced   balanced      ✓
10     balanced   balanced      ✓
-----------------------------------
All results match: True

The quantum algorithm used exactly 1 oracle query per function,
while the classical algorithm needed up to 9 queries.
